# 軽量ローカルLLMカスタマイズ実習 Notebook

この Notebook は、`docs/local-llm-customization/` の Markdown 教材を読みながら、プロンプト設計、RAG、ツール連携、LoRA / QLoRA、継続事前学習、評価と運用を小さく試すための教材です。

外部 RAG ライブラリは使わず、標準ライブラリの TF-IDF / キーワード検索で関連チャンクを探します。Ollama が未起動でも例外理由を表示し、Notebook 全体は止めません。実学習は既定で `RUN_TRAINING = False` です。

from: `docs/local-llm-customization/`

Cf. この Notebook は、docs を読み返すための小さな作業場です。汎用配布教材ではなく、この `C:\\LLM` 環境で腹落ちさせるための実験メモとして使います。

メモの凡例:
- 要確認: 手元の Ollama / GPU / パッケージ状態に依存する確認
- 未検証: この Notebook では重い実行を避け、形だけ確認する項目
- 発展: 実運用や別 Notebook に切り出すとよい項目

```mermaid
flowchart LR
  A[docsを読む] --> B[プロンプトで型を作る]
  B --> C[RAGで根拠を渡す]
  C --> D[Pythonツールで確かめる]
  D --> E[LoRA/QLoRAのデータ形を知る]
  E --> F[評価表で運用する]
```

要確認: Ollama のモデルが未取得の場合、回答生成セルは失敗理由だけを表示します。未検証: LoRA / QLoRA と継続事前学習の実学習。発展: Chroma / FAISS などへの置き換え。

## 0 導入: 設定と安全な実行ヘルパー

指定モデル名は、この repository の Ollama 運用メモに合わせます。公開ダミーデータだけを使い、ローカル専用データや未公開情報は読み込みません。

In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import re
import shutil
import statistics
import subprocess
import urllib.error
import urllib.request
from collections import Counter, defaultdict

OLLAMA_MODEL_TEXT = "batiai/gemma4-26b:iq4"
OLLAMA_MODEL_CODE = "batiai/qwen3.6-35b:iq3"
TRAINING_MODEL_ID = "google/gemma-4-E4B-it"
TRAINING_COMPARE_MODEL_ID = "Qwen/Qwen3-8B"
RUN_TRAINING = False
TOKENIZER_CHECK = False
OLLAMA_TIMEOUT_SECONDS = 90
OLLAMA_NUM_PREDICT = 512
OLLAMA_TEMPERATURE = 0.2

def find_ollama_command():
    found = shutil.which("ollama")
    if found:
        return found
    local_appdata = Path(os.environ.get("LOCALAPPDATA", ""))
    candidates = [
        local_appdata / "Programs" / "Ollama" / "ollama.exe",
        local_appdata / "Ollama" / "ollama.exe",
        Path("C:/Program Files/Ollama/ollama.exe"),
    ]
    for candidate in candidates:
        try:
            if candidate.exists():
                return str(candidate)
        except OSError:
            pass
    return None

OLLAMA_COMMAND = find_ollama_command()

GEMMA_MODEL = OLLAMA_MODEL_TEXT
QWEN_MODEL = OLLAMA_MODEL_CODE

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "docs").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
DOCS_DIR = REPO_ROOT / "docs" / "local-llm-customization"
DATA_DIR = REPO_ROOT / "notebooks" / "data"

MODEL_CONFIG = {
    "gemma": {
        "model": GEMMA_MODEL,
        "role": "説明、要約、講評、RAG回答の第一候補",
        "notes": "文章生成が軽快な想定。RAG回答の本文生成に使う。",
    },
    "qwen": {
        "model": QWEN_MODEL,
        "role": "構造化出力、技術確認、レビューの比較候補",
        "notes": "JSONや手順分解を比較するときに使う。",
    },
}
ENVIRONMENT_CARD = {
    "workspace": "C:\\LLM",
    "os_shell": "Windows / PowerShell",
    "gpu": "RTX 4060 Ti 16GB",
    "local_llm_apps": ["Ollama", "LM Studio"],
    "vram_note": "13GB級モデルを1つずつ使う。GemmaとQwenを同時常駐させない。",
    "lmstudio_note": "PDFや画像入力などGUIで確認したい時の補助環境。Notebookの自動呼び出しはOllamaに寄せる。",
}

print("repo_root:", REPO_ROOT)
print("docs_dir exists:", DOCS_DIR.exists())
print("data_dir exists:", DATA_DIR.exists())
print("ollama command:", OLLAMA_COMMAND or "not found")
print(json.dumps(ENVIRONMENT_CARD, ensure_ascii=False, indent=2))
print(json.dumps(MODEL_CONFIG, ensure_ascii=False, indent=2))

In [ ]:
OLLAMA_GENERATE_API = "http://127.0.0.1:11434/api/generate"

def post_ollama_json(payload, timeout):
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(OLLAMA_GENERATE_API, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def stop_ollama_model(model):
    """Unload one model when Ollama is available. This PC has tight 16GB VRAM."""
    try:
        post_ollama_json({"model": model, "prompt": "", "stream": False, "keep_alive": "0s"}, timeout=30)
    except Exception as exc:
        return {"ok": False, "reason": type(exc).__name__ + ": " + str(exc)}
    return {"ok": True, "reason": ""}

def run_ollama(model, prompt, timeout=OLLAMA_TIMEOUT_SECONDS, keepalive="0s"):
    """Run Ollama through the local API. Return a dict instead of raising."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "keep_alive": keepalive,
        "options": {"num_predict": OLLAMA_NUM_PREDICT, "temperature": OLLAMA_TEMPERATURE},
    }
    try:
        result = post_ollama_json(payload, timeout=timeout)
    except TimeoutError:
        return {"ok": False, "reason": f"ollama timed out after {timeout}s", "stdout": "", "stderr": ""}
    except urllib.error.URLError as exc:
        return {"ok": False, "reason": "ollama API was not reachable: " + str(exc.reason), "stdout": "", "stderr": ""}
    except Exception as exc:
        return {"ok": False, "reason": type(exc).__name__ + ": " + str(exc), "stdout": "", "stderr": ""}
    return {"ok": True, "reason": "", "stdout": (result.get("response") or "").strip(), "stderr": "", "metrics": result}

def show_ollama_result(result):
    if result["ok"]:
        print(result["stdout"])
    else:
        print("Ollama回答生成をスキップしました:", result["reason"])
        if result.get("stderr"):
            print("stderr:", result["stderr"][:1000])
        if result.get("stdout"):
            print("stdout:", result["stdout"][:1000])

def run_ollama_single_model(model, prompt, timeout=OLLAMA_TIMEOUT_SECONDS):
    """Run then unload, so Gemma and Qwen do not stay resident together."""
    result = run_ollama(model, prompt, timeout=timeout, keepalive="0s")
    stop_result = stop_ollama_model(model)
    if stop_result.get("reason") and result.get("ok"):
        result["stderr"] = (result.get("stderr") or "") + "\nstop note: " + stop_result["reason"]
    return result

## 1 全体像: どのカスタマイズを選ぶか

プロンプト、RAG、ツール連携、LoRA / QLoRA、継続事前学習は役割が違います。まずは重みを変えない方法から始め、評価で不足が見えたところだけ学習系を検討します。

#local-LLM #RTX4060Ti #Ollama #LMStudio

関連ノート: [プロンプト設計] -> [RAG] -> [ツール連携] -> [LoRA/QLoRA] -> [評価と運用]

戻りリンク: [[docs/local-llm-customization]]

In [ ]:
customization_map = [
    {"method": "プロンプト設計", "changes_weights": False, "best_for": "役割、形式、文体、判断基準を指定する"},
    {"method": "RAG", "changes_weights": False, "best_for": "更新される資料や根拠を検索して渡す"},
    {"method": "ツール連携", "changes_weights": False, "best_for": "CSV計算、ファイル検索、文献整理をPython等に任せる"},
    {"method": "LoRA / QLoRA", "changes_weights": True, "best_for": "出力形式、文体、作業手順を安定させる"},
    {"method": "継続事前学習", "changes_weights": True, "best_for": "専門分野の語彙や文体に広く慣らす"},
]
for row in customization_map:
    marker = "重み変更あり" if row["changes_weights"] else "重み変更なし"
    print(f"- {row['method']} ({marker}): {row['best_for']}")

## 2 プロンプト設計: 同じ依頼を型で変える

ここでは実行コストを抑えるため、まずプロンプトだけを比較します。Ollama が使える場合は、Gemma と Qwen に同じ課題を投げて差を見ます。

In [ ]:
task = "研究室の新入生向けに、RAGとファインチューニングの違いを説明してください。"
prompt_basic = task
prompt_structured = """
あなたは大学の研究・教育を支援するローカルLLMアシスタントです。
次の制約で回答してください。
- まず結論を1文で述べる
- RAGとファインチューニングを箇条書きで比較する
- 最後に、最初に試すべき方法を理由付きで述べる
- 根拠がないことは断定しない

依頼: {task}
""".strip().format(task=task)

print("--- basic prompt ---")
print(prompt_basic)
print("\n--- structured prompt ---")
print(prompt_structured)

In [ ]:
print("Gemma prompt comparison")
show_ollama_result(run_ollama_single_model(GEMMA_MODEL, prompt_structured))

print("\nQwen structured output comparison")
qwen_prompt = prompt_structured + "\n\n回答は JSON ではなく、読みやすいMarkdownで返してください。"
show_ollama_result(run_ollama_single_model(QWEN_MODEL, qwen_prompt))

## 3 RAG: Markdown 教材を TF-IDF / キーワード検索する

`docs/local-llm-customization/` の Markdown を見出し単位でチャンク化し、関連チャンクと出典ファイル / 見出しを表示します。検索は外部 RAG ライブラリを使わず、標準ライブラリだけで実装します。

In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_]+|[\u3040-\u30ff\u3400-\u9fff]+", text.lower())

def split_markdown_by_heading(path):
    text = path.read_text(encoding="utf-8")
    chunks = []
    current_heading = path.stem
    current_lines = []
    for line in text.splitlines():
        if line.startswith("#"):
            if current_lines:
                chunks.append({"file": path.name, "heading": current_heading, "text": "\n".join(current_lines).strip()})
            current_heading = line.lstrip("#").strip() or path.stem
            current_lines = [line]
        else:
            current_lines.append(line)
    if current_lines:
        chunks.append({"file": path.name, "heading": current_heading, "text": "\n".join(current_lines).strip()})
    return [chunk for chunk in chunks if chunk["text"]]

def load_markdown_corpus(docs_dir):
    paths = sorted(docs_dir.glob("*.md"))
    chunks = []
    for path in paths:
        chunks.extend(split_markdown_by_heading(path))
    return chunks

chunks = load_markdown_corpus(DOCS_DIR)
print("chunks:", len(chunks))
for chunk in chunks[:5]:
    print(f"- {chunk['file']} :: {chunk['heading']}")

In [ ]:
def build_tfidf_index(chunks):
    doc_terms = []
    document_frequency = Counter()
    for chunk in chunks:
        counts = Counter(tokenize(chunk["text"]))
        doc_terms.append(counts)
        document_frequency.update(counts.keys())
    n_docs = max(len(chunks), 1)
    idf = {term: math.log((1 + n_docs) / (1 + df)) + 1 for term, df in document_frequency.items()}
    vectors = []
    for counts in doc_terms:
        weighted = {term: count * idf[term] for term, count in counts.items()}
        norm = math.sqrt(sum(value * value for value in weighted.values())) or 1.0
        vectors.append({term: value / norm for term, value in weighted.items()})
    return {"idf": idf, "vectors": vectors}

def search_chunks(query, chunks, index, top_k=4):
    query_counts = Counter(tokenize(query))
    query_vector = {term: count * index["idf"].get(term, 1.0) for term, count in query_counts.items()}
    query_norm = math.sqrt(sum(value * value for value in query_vector.values())) or 1.0
    query_vector = {term: value / query_norm for term, value in query_vector.items()}
    scored = []
    for chunk, vector in zip(chunks, index["vectors"]):
        score = sum(query_vector.get(term, 0.0) * vector.get(term, 0.0) for term in query_vector)
        keyword_hits = sorted(set(query_counts) & set(tokenize(chunk["text"])))
        if score > 0 or keyword_hits:
            scored.append((score, keyword_hits, chunk))
    scored.sort(key=lambda item: (item[0], len(item[1])), reverse=True)
    return scored[:top_k]

index = build_tfidf_index(chunks)
question = "RAGとLoRAはどう使い分けるべきですか。授業資料を更新する場合を含めて説明してください。"
rag_hits = search_chunks(question, chunks, index, top_k=4)

for rank, (score, keyword_hits, chunk) in enumerate(rag_hits, start=1):
    preview = re.sub(r"\s+", " ", chunk["text"])[:240]
    print(f"[{rank}] score={score:.3f} source={chunk['file']} :: {chunk['heading']}")
    print("keywords:", ", ".join(keyword_hits[:10]) or "-")
    print(preview)
    print()

In [ ]:
def build_rag_prompt(question, hits):
    context_blocks = []
    for rank, (_, _, chunk) in enumerate(hits, start=1):
        text = chunk["text"][:1200]
        context_blocks.append(f"[資料{rank}] {chunk['file']} :: {chunk['heading']}\n{text}")
    context = "\n\n".join(context_blocks)
    return f"""
あなたは大学の研究・教育を支援するローカルLLMアシスタントです。
以下の資料抜粋だけを根拠に回答してください。
資料に書かれていないことは推測せず、「資料内では確認できません」と述べてください。
最後に、根拠として使った資料番号と出典ファイル名を列挙してください。

# 質問
{question}

# 資料抜粋
{context}
""".strip()

rag_prompt = build_rag_prompt(question, rag_hits)
print(rag_prompt[:3000])

In [ ]:
print("Gemma RAG回答")
show_ollama_result(run_ollama_single_model(GEMMA_MODEL, rag_prompt))

## 4 ツール連携: CSV 処理と構造化出力

数値計算は Python に任せ、LLM には解釈や説明を任せます。ここでは公開ダミー CSV を読み、条件ごとの平均を出し、その結果を Qwen に JSON 形式で説明させるプロンプトを作ります。

In [ ]:
csv_path = DATA_DIR / "sample_measurements.csv"
rows = list(csv.DictReader(csv_path.open(encoding="utf-8")))
scores_by_condition = defaultdict(list)
for row in rows:
    scores_by_condition[row["condition"]].append(float(row["score"]))

summary = []
for condition, scores in sorted(scores_by_condition.items()):
    summary.append({
        "condition": condition,
        "n": len(scores),
        "mean": round(statistics.mean(scores), 3),
        "min": min(scores),
        "max": max(scores),
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
structured_prompt = f"""
次のCSV集計結果を、指定スキーマのJSONだけで説明してください。
推測で原因を書かず、dummy dataであることを明記してください。

スキーマ:
{{
  "summary": "短い要約",
  "best_condition": "平均値が最も高いcondition",
  "cautions": ["注意点"],
  "next_steps": ["次に確認すること"]
}}

集計結果:
{json.dumps(summary, ensure_ascii=False, indent=2)}
""".strip()

print("Qwen構造化出力比較")
show_ollama_result(run_ollama_single_model(QWEN_MODEL, structured_prompt))

## 5 LoRA / QLoRA: 設定と小データの形を確認する

ここでは実学習を行いません。`RUN_TRAINING = False` のガード下で、LoRA / QLoRA に渡す設定と JSONL データの形だけを確認します。

In [ ]:
lora_config = {
    "base_model": TRAINING_MODEL_ID,
    "compare_model": TRAINING_COMPARE_MODEL_ID,
    "method": "QLoRA for memory saving, LoRA for simpler full precision experiments",
    "target_modules_example": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "load_in_4bit": True,
    "seq_len": 1024,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "max_steps": 20,
    "run_training": RUN_TRAINING,
}

dataset_path = DATA_DIR / "lora_dummy_dataset.jsonl"
examples = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]

print(json.dumps(lora_config, ensure_ascii=False, indent=2))
print("examples:", len(examples))
print(json.dumps(examples[0], ensure_ascii=False, indent=2))

if RUN_TRAINING:
    raise NotImplementedError("この教材Notebookでは実学習コードを直接実行しません。専用の学習環境で実行してください。")
else:
    print("RUN_TRAINING=False のため、実学習はスキップしました。")

In [ ]:
gemma_qwen_training_compare = [
    {"model": TRAINING_MODEL_ID, "fit": "説明・講評の文体寄せ", "watch": "4bit QLoRA、短いseq_len、公開ダミーデータで形式だけ確認"},
    {"model": TRAINING_COMPARE_MODEL_ID, "fit": "構造化出力・手順化・コード周辺", "watch": "同じデータ形式で比較候補にする。実ロードはこのNotebookでは前提にしない"},
]
for row in gemma_qwen_training_compare:
    print(f"- {row['model']}: 向き={row['fit']} / 注意={row['watch']}")

## 6 継続事前学習: 小コーパスと tokenizer 確認ガード

継続事前学習は RAG や LoRA より重い選択肢です。ここでは公開ダミーコーパスを読み、文字数や簡易トークンを確認します。Hugging Face tokenizer の確認は `TOKENIZER_CHECK = True` にした場合だけ試み、ローカルに無ければ理由を表示します。

In [ ]:
corpus_path = DATA_DIR / "continued_pretraining_corpus.txt"
corpus_text = corpus_path.read_text(encoding="utf-8")
simple_tokens = tokenize(corpus_text)
print("chars:", len(corpus_text))
print("simple_tokens:", len(simple_tokens))
print("unique_simple_tokens:", len(set(simple_tokens)))
print("preview:", corpus_text[:120])

TOKENIZER_MODEL_ID = TRAINING_MODEL_ID
if TOKENIZER_CHECK:
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL_ID, local_files_only=True)
        token_ids = tokenizer(corpus_text)["input_ids"]
        print("hf_tokenizer:", TOKENIZER_MODEL_ID)
        print("hf_tokens:", len(token_ids))
    except Exception as exc:
        print("tokenizer確認をスキップしました:", type(exc).__name__ + ": " + str(exc))
else:
    print("TOKENIZER_CHECK=False のため、Hugging Face tokenizer 確認はスキップしました。")

## 7 評価と運用: チェック表

ローカル LLM のカスタマイズは、作って終わりではありません。検索結果、回答、失敗例、データ境界、運用手順を小さな表で確認します。

In [ ]:
evaluation_checklist = [
    {"area": "プロンプト", "check": "同じ評価質問でGemma/Qwenの回答差を比較した", "done": False},
    {"area": "RAG検索", "check": "回答前に関連チャンクと出典ファイル/見出しを確認した", "done": False},
    {"area": "RAG回答", "check": "資料に無いことを推測せず、根拠を列挙した", "done": False},
    {"area": "ツール連携", "check": "数値計算はPythonで行い、LLMは説明に使った", "done": False},
    {"area": "LoRA/QLoRA", "check": "実学習前に小データの形式と評価観点を確認した", "done": False},
    {"area": "継続事前学習", "check": "RAG/LoRAで足りない理由と公開可能なコーパス境界を確認した", "done": False},
    {"area": "運用", "check": "個人情報、未公開情報、ログ、モデル本体をNotebookに混ぜていない", "done": True},
]

for item in evaluation_checklist:
    mark = "[x]" if item["done"] else "[ ]"
    print(f"{mark} {item['area']}: {item['check']}")